# Listening and Seeing: Whisper and MediaPipe

Iman Mossavsat

Fontys ICT

November 2026

Today you use two pretrained models: **Whisper** (speech to text) and **MediaPipe Hands** (hand tracking).

> **Important: run this notebook on your own laptop (Jupyter or VS Code), NOT in Colab.**
> Colab runs on a remote server. It cannot see your webcam in a live way, and we want to keep your voice and face on your own machine (privacy!).

As in the YOLO workshop, you do not need to understand the internals. Experiment, and find out what these models can and cannot do.

## Part 1: Whisper, turning speech into text
Whisper is a pretrained model from OpenAI. Bigger models are more accurate but slower. You will upload an audio file and pick a size.

In [ ]:
# ============================================================
# STEP 1: INSTALL (run once)
# ============================================================

# Whisper needs ffmpeg installed on your laptop to read audio files.
#   Windows: winget install ffmpeg   |  Mac: brew install ffmpeg  |  Linux: sudo apt install ffmpeg
%pip install -q openai-whisper ipywidgets

In [ ]:
# ============================================================
# STEP 2: UPLOAD A FILE AND PICK A MODEL
# ============================================================

import ipywidgets as w
from IPython.display import display

# Upload button: works with .mp3, .wav, .m4a ... (record a voice memo on your phone!)
upload = w.FileUpload(accept="audio/*,video/*", multiple=False)

# tiny = fastest, base/small = good balance, medium/large = best but slow
size = w.Dropdown(options=["tiny", "base", "small", "medium"], value="base", description="Model:")
display(upload, size)

In [ ]:
# ============================================================
# STEP 3: TRANSCRIBE
# ============================================================

import whisper

# Save the uploaded file to disk so Whisper can read it.
item = upload.value[0] if isinstance(upload.value, (list, tuple)) else list(upload.value.values())[0]
with open("input_audio", "wb") as f:
    f.write(item["content"])

model = whisper.load_model(size.value)   # downloads the model the first time
result = model.transcribe("input_audio") # language is detected automatically

print("Language:", result["language"])
print(result["text"])

# Whisper also gives timestamps per segment.
for seg in result["segments"][:5]:
    print(f"{seg['start']:.1f}s - {seg['end']:.1f}s: {seg['text']}")

### Task 1: Compare models
Transcribe the same file with `tiny` and `medium`. Compare speed and mistakes. Try another language, a noisy recording, and names or jargon.

### Feature ideas (pick one to build)
* **Voice control:** record a short command ("open notes", "next slide"), transcribe it, and use `if "next" in text:` to trigger an action.
* **Audio diary:** record daily, transcribe, and save `date + text` to a file or SQLite. Add search ("what did I say about my project?").
* **Meeting notes:** use the segment timestamps to build subtitles (.srt) or a summary with an LLM.
* **Privacy question:** the model runs locally. Why does that matter for a diary?

## Part 2: MediaPipe, hand gesture recognition
MediaPipe finds **21 landmarks** (points) on each hand in real time. From those points you can build your own gestures, for example counting raised fingers.

In [ ]:
# ============================================================
# STEP 4: INSTALL
# ============================================================

# We pin an older version because it has the simple 'solutions' API.
# It needs Python 3.9 to 3.12.
%pip install -q mediapipe==0.10.14 opencv-python

In [5]:
# ============================================================
# STEP 5: WRITE THE HAND TRACKING SCRIPT
# ============================================================
%%writefile hands.py
# STEP 5: WRITE THE HAND TRACKING SCRIPT
# We run this as a separate script (next step) because webcam windows
# crash or lag when opened from inside a notebook, especially on macOS.
import cv2, mediapipe as mp

mp_hands = mp.solutions.hands
FINGER_TIPS = [8, 12, 16, 20]   # index, middle, ring, pinky fingertip landmarks

def count_fingers(lm):
    # A finger is "up" if its tip is higher (smaller y) than the joint two points below it.
    return sum(lm[tip].y < lm[tip - 2].y for tip in FINGER_TIPS)

hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.6)
cap = cv2.VideoCapture(0)       # 0 = default webcam

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.flip(frame, 1)                                      # mirror view
    result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))  # MediaPipe expects RGB

    if result.multi_hand_landmarks:
        hand = result.multi_hand_landmarks[0]
        mp.solutions.drawing_utils.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)
        n = count_fingers(hand.landmark)
        cv2.putText(frame, f"{n} fingers", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 3)

    cv2.imshow("Hands (press q to quit)", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

UsageError: Line magic function `%%writefile` not found.


In [ ]:
# ============================================================
# STEP 6: RUN IT (a window opens, press q in it to stop)
# ============================================================
import sys
!{sys.executable} hands.py

### Task 2: Make it yours
* Define a new gesture (thumbs up, pointing, peace sign) using the landmark positions.
* Where does it fail? Try low light, a fast hand, two hands, a different skin tone or a glove.
* Idea: control volume, a game or slides with gestures. Combine with Part 1: **say** a command or **show** a gesture.

## Go further
See the full project with experiment mode and trial analysis: **[gesture_lab](https://github.com/imanmossavat/gesture_lab)**